In [1]:
import numpy as np
from helpers import load_csv_data

# Data Loading

### Load the file and convert it into numpy arrays.

In [ ]:
print("Loading subsampled data...")
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(data_path=".", sub_sample=True) #for full training later sub_sample=False

print("Data loaded successfully!")

Loading subsampled data...
Data loaded successfully!


# Data inspection

### Inspect the shape of the arrays

In [3]:
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_test shape:  {x_test.shape}")

x_train shape: (6563, 321)
y_train shape: (6563,)
x_test shape:  (109379, 321)


### Inspect the target label (y_train).

- Class -1 (healthy)

- Class +1 (diagnosed with MCHD/ heart condition)

In [4]:
unique_labels, counts = np.unique(y_train, return_counts=True)

for label, count in zip(unique_labels, counts):
  percentage = (count / len(y_train)) * 100
  print(f"Class {label}: {count} individuals ({percentage:.2f}%)")

Class -1: 6002 individuals (91.45%)
Class 1: 561 individuals (8.55%)


A model that guess -1 for everybody will achieve 91% accuracy without learning anything. 

F1-score or balanced accuracy should be considered, not just standard accuracy.

### Inspection of missing values (NaN) in the features

In [5]:
# Check total missing values
total_nans = np.isnan(x_train).sum()
print(f"Total NaNs in x_train: {total_nans}")

# Calculate percentage of missing values per column
nan_percentage_per_col = np.isnan(x_train).mean(axis=0) * 100

# Count how many columns have severe missingness (> 80% NaNs)
severely_missing_cols = np.sum(nan_percentage_per_col > 80.0)
print(f"Columns with over 80% missing data: {severely_missing_cols} / 321")

Total NaNs in x_train: 941257
Columns with over 80% missing data: 116 / 321


### Inspect feature scales and constant columns

In [6]:
# Check standard deviation of each feature
stds = np.nanstd(x_train, axis=0)

# Check if any feature has standard deviation equal to 0 (constant columns)
constant_cols = np.sum(stds == 0)
print(f"Number of constant columns (no variation): {constant_cols}")

# Inspect range of the first 5 non-empty features
for col_idx in range(5):
  min_val = np.nanmin(x_train[:, col_idx])
  max_val = np.nanmax(x_train[:, col_idx])
  print(f"Feature {col_idx}: Min = {min_val}, Max = {max_val}")

Number of constant columns (no variation): 11
Feature 0: Min = 1.0, Max = 72.0
Feature 1: Min = 1.0, Max = 12.0
Feature 2: Min = 1012016.0, Max = 12312015.0
Feature 3: Min = 1.0, Max = 12.0
Feature 4: Min = 1.0, Max = 31.0


- feature 1 is ranged between 1 and 12.
- feature 2 is ranged around 10'000'000 

This highlight the fact that we should standardize the features before performing a logistic regression.

# Data Cleaining

### Drop mostly empty columns

In [8]:
# Calculate the fraction of NaNs per column on x_train
nan_ratio = np.isnan(x_train).mean(axis=0)

# Keep only columns that have LESS than 80% NaNs
cols_to_keep = nan_ratio < 0.8

# Filter both train and test arrays using the exact same columns
x_train_filtered = x_train[:, cols_to_keep]
x_test_filtered = x_test[:, cols_to_keep]

print(f"Features kept after dropping empty columns: {x_train_filtered.shape[1]}")

Features kept after dropping empty columns: 205


### Fill ramaining NaN values

In [9]:
# 1. Compute mean of each column in x_train, ignoring NaNs
train_means = np.nanmean(x_train_filtered, axis=0)

# 2. In case any column mean is somehow NaN, replace that mean with 0
train_means = np.nan_to_num(train_means, nan=0.0)

# 3. Find where NaNs exist
train_nans = np.isnan(x_train_filtered)
test_nans = np.isnan(x_test_filtered)

# 4. Fill NaNs with the computed training means
x_train_imputed = np.where(train_nans, train_means, x_train_filtered)
x_test_imputed = np.where(test_nans, train_means, x_test_filtered)

print(
    f"Remaining NaNs in x_train: {np.isnan(x_train_imputed).sum()}"
)  # Should be 0
print(
    f"Remaining NaNs in x_test:  {np.isnan(x_test_imputed).sum()}"
)  # Should be 0

Remaining NaNs in x_train: 0
Remaining NaNs in x_test:  0


### Drop constant column

In [10]:
# Compute standard deviation per column on the imputed training set
stds = np.std(x_train_imputed, axis=0)

# Keep only columns where std > 0 (or a tiny epsilon to avoid near-zero variance)
non_constant = stds > 1e-8

x_train_clean = x_train_imputed[:, non_constant]
x_test_clean = x_test_imputed[:, non_constant]

print(f"Features remaining after dropping constants: {x_train_clean.shape[1]}")

Features remaining after dropping constants: 199


### Standardize the feature

In [11]:
mean = np.mean(x_train_clean, axis=0)
std = np.std(x_train_clean, axis=0)

# Guard against any residual zero std
std_safe = np.where(std == 0, 1.0, std)

x_train_std = (x_train_clean - mean) / std_safe
x_test_std = (x_test_clean - mean) / std_safe

print(
    f"Mean of feature 0: {np.mean(x_train_std[:, 0]):.4f}"
)  # will be close to 0
print(
    f"Std of feature 0:  {np.std(x_train_std[:, 0]):.4f}"
)  # will be close to 1

Mean of feature 0: 0.0000
Std of feature 0:  1.0000


In [ ]:
def preprocess_data(x_train, x_test, nan_threshold=0.8):
  """Cleans and standardizes the dataset using only training statistics:

  1. Drops columns with > nan_threshold missing values. 
  2. Replaces remaining NaNs using training column means. 
  3. Drops constant features (zero variance).
  4. Standardizes to zero mean and unit variance.
  """
  # 1. Filter out columns with too many NaNs
  nan_ratio = np.isnan(x_train).mean(axis=0)
  valid_cols = nan_ratio < nan_threshold
  x_train_filt = x_train[:, valid_cols]
  x_test_filt = x_test[:, valid_cols]

  # 2. Impute NaNs using training means
  train_means = np.nanmean(x_train_filt, axis=0)
  train_means = np.nan_to_num(train_means, nan=0.0)

  x_train_imp = np.where(np.isnan(x_train_filt), train_means, x_train_filt)
  x_test_imp = np.where(np.isnan(x_test_filt), train_means, x_test_filt)

  # 3. Drop constant columns
  stds = np.std(x_train_imp, axis=0)
  non_constant = stds > 1e-8
  x_train_clean = x_train_imp[:, non_constant]
  x_test_clean = x_test_imp[:, non_constant]

  # 4. Standardize using training statistics
  final_means = np.mean(x_train_clean, axis=0)
  final_stds = np.std(x_train_clean, axis=0)
  safe_stds = np.where(final_stds == 0, 1.0, final_stds)

  x_train_std = (x_train_clean - final_means) / safe_stds
  x_test_std = (x_test_clean - final_means) / safe_stds

  return x_train_std, x_test_std

### Add an intercept

In [14]:
def add_bias_term(x):
  """Add a column of 1 to the left side of the matrix features."""
  ones = np.ones((x.shape[0], 1))
  return np.hstack((ones, x))

In [15]:
tx_train = add_bias_term(x_train_std)
tx_test = add_bias_term(x_test_std)

# Ridge regression

In [20]:
from implementations import ridge_regression

# lambda_ is the regularization parameter
lambda_ = 1e-4
w_ridge, loss = ridge_regression(y_train, tx_train, lambda_)
print(f"Amount of weights w : {w_ridge.shape}")

Amount of weights w : (200,)


### Add a decision threshold (crucial for unbalanced data)

In [ ]:
y_pred_scores = tx_train @ w_ridge
print(f"Min score: {y_pred_scores.min():.3f}")
print(f"Max score: {y_pred_scores.max():.3f}")
print(f"Mean score: {y_pred_scores.mean():.3f}\n")

# Scan negative thresholds
for threshold in np.linspace(-0.9, -0.5, 9):
  y_pred = np.where(y_pred_scores >= threshold, 1, -1)

  tp = np.sum((y_train == 1) & (y_pred == 1)) #true positive
  fp = np.sum((y_train == -1) & (y_pred == 1)) #false positive
  fn = np.sum((y_train == 1) & (y_pred == -1)) #false negative

  precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
  recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0 #recall/sensitivity
  f1 = (
      2 * precision * recall / (precision + recall)
      if (precision + recall) > 0
      else 0.0
  )

  print(
      f"Threshold : {threshold:.2f} | Precision : {precision:.4f} | Recall : {recall:.4f} | F1 : {f1:.4f}"
  )

Min score: -1.423
Max score: 0.418
Mean score: -0.829

Threshold : -0.90 | Precision : 0.1463 | Recall : 0.9626 | F1 : 0.2539
Threshold : -0.85 | Precision : 0.1695 | Recall : 0.9323 | F1 : 0.2869
Threshold : -0.80 | Precision : 0.1950 | Recall : 0.9055 | F1 : 0.3209
Threshold : -0.75 | Precision : 0.2230 | Recall : 0.8627 | F1 : 0.3544
Threshold : -0.70 | Precision : 0.2553 | Recall : 0.8004 | F1 : 0.3871
Threshold : -0.65 | Precision : 0.2855 | Recall : 0.7059 | F1 : 0.4066
Threshold : -0.60 | Precision : 0.3223 | Recall : 0.6275 | F1 : 0.4259
Threshold : -0.55 | Precision : 0.3709 | Recall : 0.5579 | F1 : 0.4456
Threshold : -0.50 | Precision : 0.4201 | Recall : 0.4688 | F1 : 0.4431


The optimal tradeoff between precision and recall is at the treshold -0.55 because we obtain a F1-score of 0.4456 = 44.56%

In [24]:
def find_best_threshold(y_true, scores):
  """Scans thresholds to maximize the F1-score on {-1, 1} targets."""
  thresholds = np.linspace(-0.90, -0.40, 100)
  best_threshold = -0.55
  best_f1 = 0.0

  for t in thresholds:
    y_pred = np.where(scores >= t, 1, -1)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == -1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == -1))

    denom = 2 * tp + fp + fn
    f1 = (2 * tp) / denom if denom > 0 else 0.0

    if f1 > best_f1:
      best_f1 = f1
      best_threshold = t

  return best_threshold, best_f1